## Silver Transformations

### Data Loading

In [0]:
# ============================================================
# Bronze: container connectivity check
#
# Purpose
# -------
# - Resolve the storage account name from the cluster environment
# - Construct the ABFS(S) path for the Bronze container
# - List container contents to verify access and credentials
#
# Notes
# -----
# - This is an exploratory / validation cell
# - Failure here usually indicates:
#     * missing STORAGE_ACCOUNT_NAME
#     * incorrect cluster identity permissions
# ============================================================

from __future__ import annotations

import os

# ------------------------------------------------------------
# Resolve storage account name from environment
# ------------------------------------------------------------

storage_account: str | None = os.getenv("STORAGE_ACCOUNT_NAME")

# NOTE:
# This cell intentionally does NOT raise if the variable is missing.
# A failure at dbutils.fs.ls provides a fast, visible signal instead.
bronze_root: str = f"abfss://bronze@{storage_account}.dfs.core.windows.net/"

# ------------------------------------------------------------
# Display Bronze container contents
# ------------------------------------------------------------

display(dbutils.fs.ls(bronze_root))


path,name,size,modificationTime
abfss://bronze@stadvworksuniquegoose.dfs.core.windows.net/calendar/,calendar/,0,1767902238000
abfss://bronze@stadvworksuniquegoose.dfs.core.windows.net/customers/,customers/,0,1767902254000
abfss://bronze@stadvworksuniquegoose.dfs.core.windows.net/product_categories/,product_categories/,0,1767902273000
abfss://bronze@stadvworksuniquegoose.dfs.core.windows.net/product_subcategories/,product_subcategories/,0,1767902287000
abfss://bronze@stadvworksuniquegoose.dfs.core.windows.net/products/,products/,0,1767902307000
abfss://bronze@stadvworksuniquegoose.dfs.core.windows.net/returns/,returns/,0,1767902321000
abfss://bronze@stadvworksuniquegoose.dfs.core.windows.net/sales/,sales/,0,1767902338000
abfss://bronze@stadvworksuniquegoose.dfs.core.windows.net/territories/,territories/,0,1767902386000


In [0]:
# ============================================================
# Parameter-driven Bronze ingestion (CSV → DataFrames)
#
# Purpose
# -------
# - Load `parameters.json` from the parameters container
# - For each parameter row:
#     * derive a stable dictionary key
#     * read the referenced CSV file from Bronze
# - Store all DataFrames in a shared `dfs` dictionary
#
# Design Notes
# ------------
# - `dfs` acts as an in-memory registry for downstream enrichment
# - Parameter reloading keeps the notebook idempotent
# - CSV schema inference is acceptable at this stage (Bronze)
# ============================================================

from __future__ import annotations

import os
import re

# ------------------------------------------------------------
# Resolve required environment variable
# ------------------------------------------------------------

storage_account: str | None = os.getenv("STORAGE_ACCOUNT_NAME")
if not storage_account:
    raise ValueError("STORAGE_ACCOUNT_NAME is not set on this cluster.")

# ------------------------------------------------------------
# Load parameters.json (multiline JSON)
# ------------------------------------------------------------

params_path: str = (
    f"abfss://parameters@{storage_account}.dfs.core.windows.net/parameters.json"
)

params_df = spark.read.option("multiline", "true").json(params_path)
params = params_df.collect()

# ------------------------------------------------------------
# Helper: derive a safe dictionary key from parameter row
# ------------------------------------------------------------

def key_from_row(row):
    """
    Generate a stable, filesystem-safe key for the DataFrame registry.

    The key is derived from:
      - p_sink_folder
      - p_sink_file

    All non-alphanumeric characters are normalised to underscores.
    """
    raw = f"{row.p_sink_folder}_{row.p_sink_file}"
    raw = raw.replace("/", "_").replace(".", "_")
    return re.sub(r"[^A-Za-z0-9_]", "_", raw)

# ------------------------------------------------------------
# Read all parameter-defined CSVs into a dictionary
# ------------------------------------------------------------

dfs: dict[str, object] = {}

for row in params:
    key = key_from_row(row)
    path = (
        f"abfss://bronze@{storage_account}.dfs.core.windows.net/"
        f"{row.p_sink_folder}/{row.p_sink_file}"
    )

    dfs[key] = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(path)
    )

# ------------------------------------------------------------
# Example: inspect one loaded dataset
# ------------------------------------------------------------

display(dfs["sales_2017_AdventureWorks_Sales_2017_csv"])


OrderDate,StockDate,OrderNumber,ProductKey,CustomerKey,TerritoryKey,OrderLineItem,OrderQuantity
2017-01-01,2003-12-13,SO61285,529,23791,1,2,2
2017-01-01,2003-09-24,SO61285,214,23791,1,3,1
2017-01-01,2003-09-04,SO61285,540,23791,1,1,1
2017-01-01,2003-09-28,SO61301,529,16747,1,2,2
2017-01-01,2003-10-21,SO61301,377,16747,1,1,1
2017-01-01,2003-10-23,SO61301,540,16747,1,3,1
2017-01-01,2003-09-04,SO61269,215,11792,4,1,1
2017-01-01,2003-10-21,SO61269,229,11792,4,2,1
2017-01-01,2003-10-24,SO61286,528,11530,6,2,2
2017-01-01,2003-09-27,SO61286,536,11530,6,1,2


### Transformations

#### Calendar dataset

In [0]:
# ============================================================
# Calendar dimension enrichment
#
# Purpose
# -------
# - Detect an appropriate date column automatically
# - Parse it into a canonical `date` column
# - Derive standard calendar attributes:
#     * quarter
#     * month
#     * day of month
#
# Notes
# -----
# - This logic is intentionally defensive to handle schema drift
# - Enriched output replaces the original entry in `dfs`
# ============================================================

from __future__ import annotations

from pyspark.sql import functions as F

calendar_df = dfs["calendar_AdventureWorks_Calendar_csv"]

# ------------------------------------------------------------
# Resolve date column from common naming conventions
# ------------------------------------------------------------

date_col_candidates = [
    "Date",
    "date",
    "FullDate",
    "full_date",
    "CalendarDate",
]

date_col = next(
    (c for c in date_col_candidates if c in calendar_df.columns),
    None,
)

if not date_col:
    raise ValueError(
        f"No date column found. Columns: {calendar_df.columns}"
    )

# ------------------------------------------------------------
# Enrich calendar dataset with derived date attributes
# ------------------------------------------------------------

calendar_enriched = (
    calendar_df
    .withColumn("date", F.to_date(F.col(date_col)))
    .withColumn("quarter", F.quarter("date"))
    .withColumn("month", F.month("date"))
    .withColumn("day", F.dayofmonth("date"))
)

# Persist enriched DataFrame back into registry
dfs["calendar_AdventureWorks_Calendar_csv"] = calendar_enriched

display(calendar_enriched)


date,quarter,month,day
2015-01-01,1,1,1
2015-01-02,1,1,2
2015-01-03,1,1,3
2015-01-04,1,1,4
2015-01-05,1,1,5
2015-01-06,1,1,6
2015-01-07,1,1,7
2015-01-08,1,1,8
2015-01-09,1,1,9
2015-01-10,1,1,10


#### Customer dataset

In [0]:
# ============================================================
# Customers dimension enrichment
#
# Purpose
# -------
# - Identify first and last name columns (case-insensitive)
# - Construct a human-readable FullName column
# - Apply title-casing for presentation consistency
#
# Notes
# -----
# - Column detection is defensive to handle schema variations
# - Enriched output replaces the original entry in `dfs`
# ============================================================

from __future__ import annotations

from pyspark.sql import functions as F

customers_df = dfs["customers_AdventureWorks_Customers_csv"]

# ------------------------------------------------------------
# Resolve first / last name columns (case-insensitive)
# ------------------------------------------------------------

cols = {c.lower(): c for c in customers_df.columns}

first_col = cols.get("firstname") or cols.get("first_name")
last_col = cols.get("lastname") or cols.get("last_name")

if not first_col or not last_col:
    raise ValueError(
        f"Missing FirstName/LastName columns. "
        f"Columns: {customers_df.columns}"
    )

# ------------------------------------------------------------
# Create FullName column
# ------------------------------------------------------------

customers_enriched = (
    customers_df
    .withColumn(
        "FullName",
        F.initcap(
            F.concat_ws(" ", F.col(first_col), F.col(last_col))
        ),
    )
)

# Persist enriched DataFrame back into registry
dfs["customers_AdventureWorks_Customers_csv"] = customers_enriched

display(customers_enriched)


CustomerKey,Prefix,FirstName,LastName,BirthDate,MaritalStatus,Gender,EmailAddress,AnnualIncome,TotalChildren,EducationLevel,Occupation,HomeOwner,FullName
11000,MR.,JON,YANG,1966-04-08,M,M,jon24@adventure-works.com,"$90,000",2,Bachelors,Professional,Y,Jon Yang
11001,MR.,EUGENE,HUANG,1965-05-14,S,M,eugene10@adventure-works.com,"$60,000",3,Bachelors,Professional,N,Eugene Huang
11002,MR.,RUBEN,TORRES,1965-08-12,M,M,ruben35@adventure-works.com,"$60,000",3,Bachelors,Professional,Y,Ruben Torres
11003,MS.,CHRISTY,ZHU,1968-02-15,S,F,christy12@adventure-works.com,"$70,000",0,Bachelors,Professional,N,Christy Zhu
11004,MRS.,ELIZABETH,JOHNSON,1968-08-08,S,F,elizabeth5@adventure-works.com,"$80,000",5,Bachelors,Professional,Y,Elizabeth Johnson
11005,MR.,JULIO,RUIZ,1965-08-05,S,M,julio1@adventure-works.com,"$70,000",0,Bachelors,Professional,Y,Julio Ruiz
11007,MR.,MARCO,MEHTA,1964-05-09,M,M,marco14@adventure-works.com,"$60,000",3,Bachelors,Professional,Y,Marco Mehta
11008,MRS.,ROBIN,VERHOFF,1964-07-07,S,F,rob4@adventure-works.com,"$60,000",4,Bachelors,Professional,Y,Robin Verhoff
11009,MR.,SHANNON,CARLSON,1964-04-01,S,M,shannon38@adventure-works.com,"$70,000",0,Bachelors,Professional,N,Shannon Carlson
11010,MS.,JACQUELYN,SUAREZ,1964-02-06,S,F,jacquelyn20@adventure-works.com,"$70,000",0,Bachelors,Professional,N,Jacquelyn Suarez


### Save to Silver Container

In [0]:
# ============================================================
# Silver persistence: write datasets as Parquet
#
# Purpose
# -------
# - Reload `parameters.json` to ensure write consistency
# - For each parameter-defined dataset:
#     * retrieve the enriched DataFrame from `dfs`
#     * write it to the Silver container as Parquet
#
# Design Notes
# ------------
# - Uses overwrite mode for repeatable notebook execution
# - File naming mirrors input structure but swaps CSV → Parquet
# - This is a *batch* persistence step (non-streaming)
# ============================================================

from __future__ import annotations

import os
import re

# ------------------------------------------------------------
# Resolve required environment variable
# ------------------------------------------------------------

storage_account: str | None = os.getenv("STORAGE_ACCOUNT_NAME")
if not storage_account:
    raise ValueError("STORAGE_ACCOUNT_NAME is not set on this cluster.")

# ------------------------------------------------------------
# Reload parameters.json
# ------------------------------------------------------------

params_path: str = (
    f"abfss://parameters@{storage_account}.dfs.core.windows.net/parameters.json"
)

params_df = spark.read.option("multiline", "true").json(params_path)
params = params_df.collect()

# ------------------------------------------------------------
# Helper: stable dictionary key
# ------------------------------------------------------------

def key_from_row(row):
    """
    Generate the same stable key used during ingestion.
    """
    raw = f"{row.p_sink_folder}_{row.p_sink_file}"
    raw = raw.replace("/", "_").replace(".", "_")
    return re.sub(r"[^A-Za-z0-9_]", "_", raw)

# ------------------------------------------------------------
# Helper: convert CSV filename to Parquet filename
# ------------------------------------------------------------

def parquet_name(filename: str) -> str:
    """
    Replace a trailing `.csv` (case-insensitive) with `.parquet`.
    """
    return re.sub(r"(?i)\.csv$", ".parquet", filename)

# ------------------------------------------------------------
# Write each dataset to Silver
# ------------------------------------------------------------

for row in params:
    key = key_from_row(row)
    df = dfs[key]

    silver_path = (
        f"abfss://silver@{storage_account}.dfs.core.windows.net/"
        f"{row.p_sink_folder}/{parquet_name(row.p_sink_file)}"
    )

    df.write.mode("overwrite").parquet(silver_path)
